# Athena · RSNA development benchmark

This notebook runs the median baseline, ResNet-18, and ViT-B/16 on the same saved 80/20 split. The separate RSNA validation archive is reserved for final evaluation.

Select **Runtime → Change runtime type → GPU**. Colab GPU access depends on availability and limits. Leave this notebook's outputs private while running it. The official training archive alone is about 9.6 GB and needs additional space to extract.

Data source: [RSNA Pediatric Bone Age Challenge](https://www.rsna.org/artificial-intelligence/ai-image-challenge/rsna-pediatric-bone-age-challenge-2017) · [RSNA data terms](https://www.rsna.org/-/media/files/rsna/education/ai-resources-and-training/ai-image-challenge/rsna-2017-ai-challenge-terms-of-use-and-attribution_final.pdf)

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before downloading the dataset.")

repo = Path("/content/Athena")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/n1sh032/Athena.git", str(repo)], check=True)

def run(*args):
    subprocess.run([str(arg) for arg in args], cwd=repo, check=True)

run("git", "rev-parse", "HEAD")
print(torch.cuda.get_device_name(), torch.__version__, torchvision.__version__)

## Download and check the training data

The download is resumable within the same runtime. Each completed archive is kept until the runtime ends. The filenames and annotations are checked before either model trains.

In [ ]:
import csv

source = "https://s3.amazonaws.com/east1.public.rsna.org/AI/2017/"
raw = repo / "data/raw"
raw.mkdir(parents=True, exist_ok=True)

def download(remote_name, local_name):
    target = raw / local_name
    incomplete = target.with_suffix(".part")
    if not target.exists():
        run("curl", "--fail", "--location", "--retry", "3", "--continue-at", "-",
            "--output", incomplete, source + remote_name)
        incomplete.replace(target)
    return target

labels = raw / "train.csv"
if not labels.exists():
    archive = download("Bone+Age+Training+Set+Annotations.zip", "labels.zip")
    with ZipFile(archive) as contents:
        csvs = [name for name in contents.namelist() if Path(name).name == "train.csv"]
        if len(csvs) != 1:
            raise ValueError("Expected one train.csv file in the RSNA annotations archive.")
        labels.write_bytes(contents.read(csvs[0]))

with labels.open(encoding="utf-8-sig", newline="") as file:
    expected = {f"{int(row['id'])}.png" for row in csv.DictReader(file)}

images = raw / "images"
images.mkdir(exist_ok=True)
if not all((images / name).is_file() for name in expected):
    archive_path = raw / "images.zip"
    partial_path = raw / "images.part"
    if not archive_path.exists():
        partial_size = partial_path.stat().st_size if partial_path.exists() else 0
        remaining = max(0, 9_799_344_140 - partial_size)
        if shutil.disk_usage(raw).free < remaining:
            raise OSError(f"The remaining archive needs {remaining / 1e9:.1f} GB.")
    archive = download("Bone+Age+Training+Set.zip", "images.zip")
    with ZipFile(archive) as contents:
        members = {}
        for member in contents.infolist():
            name = Path(member.filename).name
            if name in expected:
                if name in members:
                    raise ValueError(f"Duplicate image in archive: {name}")
                members[name] = member
        if members.keys() != expected:
            raise ValueError("The archive image IDs do not match the training labels.")

        needed = sum(member.file_size for name, member in members.items()
                     if not (images / name).is_file())
        if shutil.disk_usage(raw).free < needed:
            raise OSError(f"Extraction needs {needed / 1e9:.1f} GB more free disk space.")

        for name, member in members.items():
            target = images / name
            if not target.exists():
                partial = target.with_suffix(".part")
                with contents.open(member) as src, partial.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
                partial.replace(target)

run(sys.executable, "scripts/check_data.py", labels, images)
run(sys.executable, "scripts/baseline.py", labels)
run(sys.executable, "-m", "scripts.image_data", labels, images)

## Train both models

Use the same batch size for both. Start at 8 to reduce GPU memory use. If ViT runs out of memory, lower the batch size and change the run name so both models rerun with the new settings. Save each model's ZIP from the Colab Files panel after its cell finishes. The ZIP contains the trained head and run report.

In [ ]:
batch_size = 8
epochs = 5
run_name = "rsna-dev-b8"

def train(module, model):
    output = repo / "artifacts" / run_name / model
    run(sys.executable, "-m", module, labels, images, "--epochs", epochs,
        "--batch-size", batch_size, "--out", output)
    bundle = shutil.make_archive(f"/content/{run_name}-{model}", "zip", output)
    print("Save from the Files panel:", bundle)

In [ ]:
train("scripts.train_resnet", "resnet18")

In [ ]:
train("scripts.train_vit", "vit_b_16")